# 04. 저장된 예측표에서 결과 표와 그림 다시 만들기

이 노트북은 학습을 하지 않습니다. Git에 들어 있는 out-of-fold 예측표(`reports/pathway_sensitivity_a1/oof/`)와 요약표(`reports/pathway_sensitivity_a1/tables/`)를 읽습니다. 예측표에는 예측값만 있으므로 실측 AA는 ID로 다시 붙입니다 (서버의 입력 캐시가 필요합니다).

In [ ]:
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "configs", "pathway_sensitivity_a1")):
    parent = os.path.dirname(root)
    assert parent != root, "저장소 안에서 실행해 주세요."
    root = parent
sys.path.insert(0, os.path.join(root, "src", "pathway_sensitivity_a1"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import core
cfg = core.load_config()
print("저장소:", root)
print("입력 캐시:", core.cache_dir(cfg))
import metrics, json
tables = os.path.join(core.REPORTS, 'tables')
runs = json.load(open(os.path.join(core.REPORTS, 'oof', 'RUNS.json')))
print(runs)

## 1. 주지표를 예측표에서 다시 계산

In [ ]:
reg = core.load_registry(); family_of = dict(zip(reg.model_id, reg.family_id))
long = core.load_limore_response(cfg); sp = core.load_outer_splits()
repeats, fams, drugs = sorted(sp.repeat.unique()), sorted(sp.family_id.unique()), sorted(long.drug_id.unique())
main = os.path.join(core.REPORTS, "oof", runs["main"])
a0 = metrics.load_oof(main, "A0", 3)
s_ref = a0[["repeat", "outer_fold", "drug_id", "s_train"]].drop_duplicates()
cube = {}
for name in ["A0", "A1", "A1r", "A2", "A3"]:
    df = metrics.attach_truth(metrics.load_oof(main, name, 3), long, family_of, s_ref)
    cube[name] = metrics.error_cube(df, repeats, drugs, fams)
mine = pd.Series({k: metrics.macro_nrmse(v) for k, v in cube.items()}, name="다시 계산")
saved = pd.read_csv(os.path.join(tables, "internal_summary.tsv"), sep="\t").set_index("model")
print(pd.concat([mine, saved.macro_drug_normalized_rmse.rename("저장된 표")], axis=1))
assert np.allclose(mine, saved.macro_drug_normalized_rmse.loc[mine.index])

## 2. 계열 bootstrap으로 차이의 구간 다시 계산

계열을 복원추출하고, 뽑힌 계열의 모든 약물·반복 예측을 함께 움직입니다.

In [ ]:
W = metrics.bootstrap_weights(len(fams), cfg["bootstrap"]["n"], cfg["bootstrap"]["seed"])
for a, b in [("A3", "A0"), ("A3", "A1"), ("A3", "A1r"), ("A3", "A2")]:
    d = metrics.boot_macro(cube[a], W) - metrics.boot_macro(cube[b], W)
    print(f"{a} - {b}: {metrics.macro_nrmse(cube[a]) - metrics.macro_nrmse(cube[b]):+.4f}  95% 구간 [{np.quantile(d, .025):+.4f}, {np.quantile(d, .975):+.4f}]")

## 3. 대조와 외부 평가 표

In [ ]:
for f in ["internal_differences.tsv", "learning_curve.tsv", "external_summary.tsv"]:
    print("\n==", f); print(pd.read_csv(os.path.join(tables, f), sep="\t").to_string(index=False))
print(json.dumps(json.load(open(os.path.join(tables, "controls_results.json")))["a4"], indent=1))

## 4. 그림 다시 만들기

결과는 `reports/pathway_sensitivity_a1/figures_regenerated/`에 저장됩니다. 저장소에 있는 그림을 덮어쓰지 않습니다.

In [ ]:
import make_figures as mf
out = os.path.join(core.REPORTS, "figures_regenerated"); os.makedirs(out, exist_ok=True)
mf.fig_internal(out, tables); mf.fig_learning(out, tables); mf.fig_a4(out, tables); mf.fig_external(out, tables)
mf.fig_cases(out, tables, os.path.join(main, "oof"))
print(sorted(os.listdir(out)))